# Enhanced Preprocessing Pipeline
## Amazon Reviews Data Processing

**Features:**
- Cross-platform support (macOS & Windows)
- Improved error handling and logging
- Better code organization and documentation
- Configuration management
- Data validation checks

## 1. Setup & Configuration

In [1]:
# === CROSS-PLATFORM SETUP ===
import subprocess
import sys
import os
import platform
from pathlib import Path

# Detect OS
SYSTEM = platform.system()
IS_WINDOWS = SYSTEM == "Windows"
IS_MAC = SYSTEM == "Darwin"
IS_LINUX = SYSTEM == "Linux"

print(f"✓ Running on {SYSTEM}")
print(f"  Python executable: {sys.executable}")

✓ Running on Darwin
  Python executable: /Library/Developer/CommandLineTools/usr/bin/python3


In [ ]:
# === INSTALL DEPENDENCIES ===
packages = [
    "datasets==3.6.0",
    "huggingface_hub==0.34.4",
    "pandas",
    "numpy",
    "pyarrow",
    "tqdm",
    "sentence-transformers"
]

print("Checking packages...")
missing_packages = []

for pkg in packages:
    pkg_name = pkg.split('==')[0].replace('-', '_').replace('.', '_')
    try:
        __import__(pkg_name)
        print(f"  ✓ {pkg.split('==')[0]} already installed")
    except ImportError:
        missing_packages.append(pkg)
        print(f"  ⚠ {pkg.split('==')[0]} missing - will install")

if missing_packages:
    print(f"\nInstalling {len(missing_packages)} packages...")
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed"] + missing_packages
    subprocess.check_call(cmd)
    print("✓ Installation complete!")
else:
    print("\n✓ All packages already installed!")

Checking packages...


/Users/mac/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/mac/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✓ datasets already installed
  ✓ huggingface_hub already installed
  ✓ pandas already installed
  ✓ numpy already installed
  ✓ pyarrow already installed
  ⚠ scikit-learn missing - will install
  ✓ tqdm already installed


In [ ]:
# === IMPORT LIBRARIES ===
import json
import re
import hashlib
import warnings
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import logging

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset

warnings.filterwarnings("ignore")

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ All libraries imported successfully")

In [ ]:
# === CONFIGURATION CLASS ===
class PreprocessingConfig:
    """Central configuration for preprocessing pipeline."""
    
    def __init__(self, project_base: Optional[Path] = None):
        # Auto-detect project base if not specified
        if project_base is None:
            # Try to find the project directory
            current_dir = Path.cwd()
            if (current_dir.name == "MXH FINAL" or "MXH FINAL" in str(current_dir)):
                project_base = current_dir
            else:
                # Default to home directory + Downloads/MXH FINAL
                project_base = Path.home() / "Downloads" / "MXH FINAL"
        
        self.PROJECT_DIR = Path(project_base)
        self.PROJECT_DIR.mkdir(parents=True, exist_ok=True)
        
        # Data directories
        self.DATA_DIR = self.PROJECT_DIR / "data"
        self.RAW_DIR = self.DATA_DIR / "raw"
        self.SILVER_DIR = self.DATA_DIR / "silver"
        self.GOLD_DIR = self.DATA_DIR / "gold"
        self.CACHE_DIR = self.PROJECT_DIR / "hf_cache"
        
        # Create directories
        for folder in [self.RAW_DIR, self.SILVER_DIR, self.GOLD_DIR, self.CACHE_DIR]:
            folder.mkdir(parents=True, exist_ok=True)
        
        # Dataset configuration
        self.CATEGORY = "All_Beauty"
        self.DATASET_NAME = "McAuley-Lab/Amazon-Reviews-2023"
        
        # Processing configuration
        self.SAMPLE_SIZE = None  # None = Full dataset | Set to e.g. 100000 for quick testing
        self.GENERATE_EMBEDDINGS = True  # True = Generate embeddings | False = Skip embeddings
        self.EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
        self.EMBEDDING_BATCH_SIZE = 64
        
        # Random state for reproducibility
        self.RANDOM_STATE = 42
        
        # Min words for reviews
        self.MIN_REVIEW_WORDS = 3
        
        # Split ratios
        self.TRAIN_RATIO = 0.7
        self.VAL_RATIO = 0.15
        self.TEST_RATIO = 0.15
    
    def display_config(self):
        """Display current configuration."""
        print("\n" + "="*50)
        print("CONFIGURATION SUMMARY")
        print("="*50)
        print(f"OS: {SYSTEM}")
        print(f"Project Directory: {self.PROJECT_DIR}")
        print(f"Category: {self.CATEGORY}")
        print(f"Sample Size: {self.SAMPLE_SIZE if self.SAMPLE_SIZE else 'Full dataset'}")
        print(f"Generate Embeddings: {self.GENERATE_EMBEDDINGS}")
        print(f"Cache Directory: {self.CACHE_DIR}")
        print("="*50 + "\n")

# Initialize configuration
config = PreprocessingConfig()
config.display_config()

## 2. Data Loading Functions

In [ ]:
def load_hf_config_to_pandas(dataset_name: str, config_name: str, cache_dir: Path) -> pd.DataFrame:
    """
    Load HuggingFace dataset configuration to pandas DataFrame.
    Handles both 'full' split and DatasetDict cases.
    """
    try:
        ds = load_dataset(
            dataset_name,
            config_name,
            split="full",
            trust_remote_code=True,
            cache_dir=str(cache_dir)
        )
        logger.info(f"Loaded {config_name} with split='full'")
        return ds.to_pandas()

    except Exception as e:
        logger.warning(f"Could not load with split='full': {str(e)[:100]}")
        logger.info(f"Trying to load {config_name} as DatasetDict...")

        ds_dict = load_dataset(
            dataset_name,
            config_name,
            trust_remote_code=True,
            cache_dir=str(cache_dir)
        )

        first_split = list(ds_dict.keys())[0]
        logger.info(f"Using split: {first_split}")

        return ds_dict[first_split].to_pandas()

def load_amazon_reviews(category: str, config: PreprocessingConfig) -> pd.DataFrame:
    """Load Amazon reviews for specified category."""
    config_name = f"raw_review_{category}"
    return load_hf_config_to_pandas(config.DATASET_NAME, config_name, config.CACHE_DIR)

def load_amazon_metadata(category: str, config: PreprocessingConfig) -> pd.DataFrame:
    """Load Amazon metadata for specified category."""
    config_name = f"raw_meta_{category}"
    return load_hf_config_to_pandas(config.DATASET_NAME, config_name, config.CACHE_DIR)

# Load reviews
logger.info("Loading Amazon reviews...")
reviews_raw = load_amazon_reviews(config.CATEGORY, config)
logger.info(f"Loaded {len(reviews_raw)} reviews")
print(f"\nRaw reviews shape: {reviews_raw.shape}")
print(f"Columns: {reviews_raw.columns.tolist()}")

In [ ]:
# Apply sampling if configured
if config.SAMPLE_SIZE is not None and len(reviews_raw) > config.SAMPLE_SIZE:
    logger.info(f"Sampling {config.SAMPLE_SIZE} reviews...")
    reviews_raw = reviews_raw.sample(n=config.SAMPLE_SIZE, random_state=config.RANDOM_STATE).reset_index(drop=True)
    logger.info(f"After sampling: {reviews_raw.shape}")

print(reviews_raw.head())

## 3. Data Standardization

In [ ]:
def convert_timestamp(series: pd.Series) -> pd.Series:
    """
    Convert Amazon timestamp (Unix ms or seconds) to datetime.
    Handles both millisecond and second formats.
    """
    s = pd.to_numeric(series, errors="coerce")

    if s.dropna().empty:
        return pd.to_datetime(series, errors="coerce")

    median_value = s.dropna().median()

    # If median > 10^12, likely milliseconds
    if median_value > 10**12:
        return pd.to_datetime(s, unit="ms", errors="coerce")
    else:
        return pd.to_datetime(s, unit="s", errors="coerce")

def make_review_id(row: pd.Series) -> str:
    """
    Create stable review_id from multiple fields.
    Ensures uniqueness and reproducibility.
    """
    raw = f"{row['user_id']}|{row['item_id']}|{row['timestamp']}|{row['rating']}|{row['review_text']}"
    return hashlib.md5(raw.encode("utf-8")).hexdigest()

def standardize_reviews(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize review data to common schema:
    - review_id, user_id, item_id, rating, review_title, review_text,
    - timestamp, helpful_vote, verified_purchase
    """
    df = df.copy()
    logger.info("Standardizing reviews...")

    # Standardize item_id
    if "parent_asin" in df.columns:
        df["item_id"] = df["parent_asin"]
    elif "asin" in df.columns:
        df["item_id"] = df["asin"]
    else:
        raise ValueError("No asin or parent_asin column found.")

    # Standardize review_title
    df["review_title"] = df.get("title", "").fillna("").astype(str)

    # Standardize review_text
    if "text" in df.columns:
        df["review_text"] = df["text"].fillna("").astype(str)
    else:
        raise ValueError("No text column found.")

    # Standardize rating
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

    # Standardize timestamp
    if "timestamp" in df.columns:
        df["timestamp"] = convert_timestamp(df["timestamp"])
    else:
        raise ValueError("No timestamp column found.")

    # Standardize helpful_vote
    df["helpful_vote"] = (
        df.get("helpful_vote", 0)
        .pipe(pd.to_numeric, errors="coerce")
        .fillna(0)
        .astype(int)
    )

    # Standardize verified_purchase
    df["verified_purchase"] = df.get("verified_purchase", False).fillna(False).astype(bool)

    keep_cols = [
        "user_id", "item_id", "rating", "review_title", "review_text",
        "timestamp", "helpful_vote", "verified_purchase"
    ]

    df = df[keep_cols]
    logger.info(f"Standardized shape: {df.shape}")

    return df

# Standardize
reviews = standardize_reviews(reviews_raw)
print(reviews.head())

## 4. Data Cleaning

In [ ]:
def clean_reviews(df: pd.DataFrame, min_words: int = 3) -> pd.DataFrame:
    """
    Clean review data:
    - Remove missing critical fields
    - Remove very short reviews
    - Remove invalid ratings (not 1-5)
    - Remove duplicates
    - Create review_id
    """
    df = df.copy()
    logger.info(f"Cleaning {len(df)} reviews...")
    initial_count = len(df)

    # Remove missing critical fields
    required_cols = ["user_id", "item_id", "rating", "review_text", "timestamp"]
    df = df.dropna(subset=required_cols)
    logger.info(f"After removing missing fields: {len(df)} reviews")

    # Clean strings
    df["user_id"] = df["user_id"].astype(str).str.strip()
    df["item_id"] = df["item_id"].astype(str).str.strip()
    df["review_text"] = df["review_text"].astype(str).str.strip()
    df["review_title"] = df["review_title"].astype(str).str.strip()

    # Remove empty strings
    df = df[(df["user_id"] != "") & (df["item_id"] != "") & (df["review_text"] != "")]
    logger.info(f"After removing empty strings: {len(df)} reviews")

    # Remove invalid ratings
    df = df[df["rating"].between(1, 5)]
    logger.info(f"After removing invalid ratings: {len(df)} reviews")

    # Remove short reviews
    df["word_count_temp"] = df["review_text"].str.split().str.len()
    df = df[df["word_count_temp"] >= min_words]
    df = df.drop(columns=["word_count_temp"])
    logger.info(f"After removing short reviews (<{min_words} words): {len(df)} reviews")

    # Remove duplicates
    df = df.drop_duplicates(
        subset=["user_id", "item_id", "timestamp", "rating", "review_text"]
    )
    logger.info(f"After removing duplicates: {len(df)} reviews")

    # Create review_id
    df["review_id"] = df.apply(make_review_id, axis=1)

    cols = [
        "review_id", "user_id", "item_id", "rating", "review_title", "review_text",
        "timestamp", "helpful_vote", "verified_purchase"
    ]

    df = df[cols].reset_index(drop=True)

    removed = initial_count - len(df)
    logger.info(f"Removed {removed} reviews ({removed/initial_count*100:.1f}%)")

    return df

# Clean reviews
reviews_clean = clean_reviews(reviews, min_words=config.MIN_REVIEW_WORDS)
print(f"\nClean reviews shape: {reviews_clean.shape}")
print(reviews_clean.head())

## 5. Text Processing

In [ ]:
def normalize_text_for_tfidf(text: str) -> str:
    """
    Light text normalization for TF-IDF or statistics.
    Does NOT use this for transformer embeddings.
    """
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)  # Remove URLs
    text = re.sub(r"[^a-zA-Z0-9\s!?.]", " ", text)  # Keep only alphanumeric and punctuation
    text = re.sub(r"\s+", " ", text).strip()  # Normalize spaces
    return text

def add_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add raw and cleaned text columns."""
    df = df.copy()
    logger.info("Adding text columns...")

    df["text_raw"] = (
        df["review_title"].fillna("").astype(str).str.strip()
        + " "
        + df["review_text"].fillna("").astype(str).str.strip()
    ).str.strip()

    df["text_clean"] = df["text_raw"].apply(normalize_text_for_tfidf)

    return df

reviews_clean = add_text_columns(reviews_clean)
print(reviews_clean[["review_id", "text_raw", "text_clean"]].head())

## 6. Text Features

In [ ]:
def count_uppercase_words(text: str) -> int:
    """Count words that are entirely uppercase (length > 1)."""
    words = str(text).split()
    return sum(1 for w in words if len(w) > 1 and w.isupper())

def add_text_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract text features:
    - Length features (chars, words)
    - Punctuation features (!, ?)
    - Case features (uppercase words)
    - Duplicate features
    """
    df = df.copy()
    logger.info("Adding text features...")

    df["text_length_chars"] = df["text_raw"].str.len()
    df["text_length_words"] = df["text_raw"].str.split().str.len()
    df["num_exclamation"] = df["text_raw"].str.count("!")
    df["num_question"] = df["text_raw"].str.count(r"\?")
    df["num_uppercase_words"] = df["text_raw"].apply(count_uppercase_words)

    df["uppercase_word_ratio"] = (
        df["num_uppercase_words"] / df["text_length_words"].replace(0, np.nan)
    ).fillna(0)

    df["is_short_review"] = (df["text_length_words"] <= 8).astype(int)
    df["is_very_short_review"] = (df["text_length_words"] <= 4).astype(int)

    df["text_fingerprint"] = (
        df["text_clean"]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    df["duplicate_text_count"] = df.groupby("text_fingerprint")["review_id"].transform("count")

    return df

reviews_clean = add_text_features(reviews_clean)
print(reviews_clean[["review_id", "text_length_words", "is_short_review", "duplicate_text_count"]].head())

## 7. User Features

In [ ]:
def add_user_features(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Add user-level aggregated features:
    - Review count, item count
    - Rating statistics
    - Temporal features
    - Verification ratio
    - Activity metrics
    """
    df = df.copy()
    logger.info("Adding user features...")

    user_agg = df.groupby("user_id").agg(
        user_review_count=("review_id", "count"),
        user_item_count=("item_id", "nunique"),
        user_avg_rating=("rating", "mean"),
        user_rating_std=("rating", "std"),
        user_first_review_time=("timestamp", "min"),
        user_last_review_time=("timestamp", "max"),
        user_verified_ratio=("verified_purchase", "mean"),
        user_avg_text_length=("text_length_words", "mean"),
    ).reset_index()

    user_agg["user_rating_std"] = user_agg["user_rating_std"].fillna(0)

    user_agg["user_active_days"] = (
        user_agg["user_last_review_time"] - user_agg["user_first_review_time"]
    ).dt.days + 1

    user_agg["user_reviews_per_day"] = (
        user_agg["user_review_count"] / user_agg["user_active_days"].replace(0, 1)
    )

    user_agg["user_singleton"] = (user_agg["user_review_count"] == 1).astype(int)

    df = df.merge(user_agg, on="user_id", how="left")

    logger.info(f"Created {len(user_agg)} user nodes")

    return df, user_agg

reviews_clean, users_clean = add_user_features(reviews_clean)
print(f"Users: {len(users_clean)}")
print(users_clean.head())

## 8. Item Features

In [ ]:
def add_item_features(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Add item-level aggregated features:
    - Review count, user count
    - Rating statistics
    - Temporal features
    - Verification ratio
    - Activity metrics
    """
    df = df.copy()
    logger.info("Adding item features...")

    item_agg = df.groupby("item_id").agg(
        item_review_count=("review_id", "count"),
        item_user_count=("user_id", "nunique"),
        item_avg_rating=("rating", "mean"),
        item_rating_std=("rating", "std"),
        item_first_review_time=("timestamp", "min"),
        item_last_review_time=("timestamp", "max"),
        item_verified_ratio=("verified_purchase", "mean"),
        item_avg_text_length=("text_length_words", "mean"),
    ).reset_index()

    item_agg["item_rating_std"] = item_agg["item_rating_std"].fillna(0)

    item_agg["item_active_days"] = (
        item_agg["item_last_review_time"] - item_agg["item_first_review_time"]
    ).dt.days + 1

    item_agg["item_reviews_per_day"] = (
        item_agg["item_review_count"] / item_agg["item_active_days"].replace(0, 1)
    )

    df = df.merge(item_agg, on="item_id", how="left")

    logger.info(f"Created {len(item_agg)} item nodes")

    return df, item_agg

reviews_clean, items_clean = add_item_features(reviews_clean)
print(f"Items: {len(items_clean)}")
print(items_clean.head())

## 9. Temporal Features

In [ ]:
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add temporal features:
    - Date, year, month, day of week
    - Review counts by time periods
    """
    df = df.copy()
    logger.info("Adding temporal features...")

    df["review_date"] = df["timestamp"].dt.date.astype(str)
    df["review_year"] = df["timestamp"].dt.year
    df["review_month"] = df["timestamp"].dt.month
    df["review_year_month"] = df["timestamp"].dt.to_period("M").astype(str)
    df["review_dayofweek"] = df["timestamp"].dt.dayofweek

    df["user_day_review_count"] = df.groupby(["user_id", "review_date"])["review_id"].transform("count")
    df["item_day_review_count"] = df.groupby(["item_id", "review_date"])["review_id"].transform("count")
    df["item_month_review_count"] = df.groupby(["item_id", "review_year_month"])["review_id"].transform("count")
    df["item_rating_month_count"] = df.groupby(
        ["item_id", "rating", "review_year_month"]
    )["review_id"].transform("count")

    return df

reviews_clean = add_temporal_features(reviews_clean)
print(reviews_clean[["review_id", "review_date", "review_month", "user_day_review_count"]].head())

## 10. Rating Deviation Features

In [ ]:
def add_rating_deviation_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add rating deviation features:
    - Deviation from item average
    - Deviation from user average
    - Extreme rating indicator
    """
    df = df.copy()
    logger.info("Adding rating deviation features...")

    df["rating_deviation_from_item_avg"] = (
        df["rating"] - df["item_avg_rating"]
    ).abs()

    df["rating_deviation_from_user_avg"] = (
        df["rating"] - df["user_avg_rating"]
    ).abs()

    df["is_extreme_rating"] = df["rating"].isin([1, 5]).astype(int)

    return df

reviews_clean = add_rating_deviation_features(reviews_clean)
print(reviews_clean[["review_id", "rating", "rating_deviation_from_item_avg", "is_extreme_rating"]].head())

## 11. Weak Labels (Suspicion Scoring)

In [ ]:
def add_weak_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create suspicion score and weak labels.
    weak_label = 1 means review shows suspicious behavior.
    This is NOT a real fake label, just weak indicators.
    """
    df = df.copy()
    logger.info("Creating weak labels...")

    score = np.zeros(len(df))

    # Rule 1: Short review + extreme rating
    score += ((df["is_short_review"] == 1) & (df["is_extreme_rating"] == 1)).astype(int)

    # Rule 2: Highly duplicated text
    score += (df["duplicate_text_count"] >= 3).astype(int)

    # Rule 3: User wrote many reviews same day
    score += (df["user_day_review_count"] >= 3).astype(int)

    # Rule 4: Item received many reviews same day
    score += (df["item_day_review_count"] >= 5).astype(int)

    # Rule 5: Many reviews same rating in same month
    score += (df["item_rating_month_count"] >= 5).astype(int)

    # Rule 6: Not verified + extreme rating
    score += ((df["verified_purchase"] == False) & (df["is_extreme_rating"] == 1)).astype(int)

    # Rule 7: New user + many reviews
    score += ((df["user_active_days"] <= 3) & (df["user_review_count"] >= 3)).astype(int)

    # Rule 8: No helpful votes + short + extreme
    score += (
        (df["helpful_vote"] == 0)
        & (df["is_short_review"] == 1)
        & (df["is_extreme_rating"] == 1)
    ).astype(int)

    df["suspicion_score"] = score
    df["weak_label"] = (df["suspicion_score"] >= 2).astype(int)

    return df

reviews_clean = add_weak_labels(reviews_clean)
print("\nWeak label distribution:")
print(reviews_clean["weak_label"].value_counts())
print("\nSuspicion score distribution:")
print(reviews_clean["suspicion_score"].value_counts().sort_index())

## 12. Train/Val/Test Split

In [ ]:
def add_time_based_split(df: pd.DataFrame, train_ratio: float = 0.7, val_ratio: float = 0.15) -> pd.DataFrame:
    """
    Create time-based train/val/test split.
    Maintains temporal order of reviews.
    """
    df = df.copy()
    logger.info("Creating time-based split...")
    df = df.sort_values("timestamp").reset_index(drop=True)

    n = len(df)
    train_end = int(train_ratio * n)
    val_end = int((train_ratio + val_ratio) * n)

    df["split"] = "test"
    df.loc[:train_end - 1, "split"] = "train"
    df.loc[train_end:val_end - 1, "split"] = "val"

    logger.info(f"Split distribution: {df['split'].value_counts().to_dict()}")

    return df

reviews_clean = add_time_based_split(reviews_clean, config.TRAIN_RATIO, config.VAL_RATIO)
print(reviews_clean["split"].value_counts())

## 13. Build Graph Nodes

In [ ]:
def build_node_tables(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build graph node tables:
    - User nodes
    - Review nodes
    - Item nodes
    """
    logger.info("Building node tables...")

    nodes_review = df[[
        "review_id", "user_id", "item_id", "rating", "timestamp", "text_raw", "text_clean",
        "helpful_vote", "verified_purchase", "suspicion_score", "weak_label", "split"
    ]].drop_duplicates("review_id").reset_index(drop=True)

    nodes_user = df[[
        "user_id", "user_review_count", "user_item_count", "user_avg_rating",
        "user_rating_std", "user_active_days", "user_reviews_per_day",
        "user_verified_ratio", "user_avg_text_length", "user_singleton"
    ]].drop_duplicates("user_id").reset_index(drop=True)

    nodes_item = df[[
        "item_id", "item_review_count", "item_user_count", "item_avg_rating",
        "item_rating_std", "item_active_days", "item_reviews_per_day",
        "item_verified_ratio", "item_avg_text_length"
    ]].drop_duplicates("item_id").reset_index(drop=True)

    logger.info(f"User nodes: {len(nodes_user)}")
    logger.info(f"Review nodes: {len(nodes_review)}")
    logger.info(f"Item nodes: {len(nodes_item)}")

    return nodes_user, nodes_review, nodes_item

nodes_user, nodes_review, nodes_item = build_node_tables(reviews_clean)
print(f"\nUser nodes: {len(nodes_user)}")
print(f"Review nodes: {len(nodes_review)}")
print(f"Item nodes: {len(nodes_item)}")

## 14. Build Graph Edges

In [ ]:
def create_consecutive_edges(
    df: pd.DataFrame,
    group_cols: List[str],
    src_col: str = "review_id",
    time_col: str = "timestamp",
    edge_type: str = "relation"
) -> pd.DataFrame:
    """
    Create edges between consecutive reviews in same group.
    More efficient than full clique.
    """
    edges = []
    grouped = df.sort_values(time_col).groupby(group_cols)

    for _, group in tqdm(grouped, desc=f"Building {edge_type} edges", leave=False):
        ids = group[src_col].tolist()

        if len(ids) < 2:
            continue

        for i in range(len(ids) - 1):
            edges.append({"src": ids[i], "dst": ids[i + 1], "edge_type": edge_type})

    return pd.DataFrame(edges)

def build_edge_tables(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """
    Build edge tables:
    - User writes Review
    - Review about Item
    - User rates Item
    - Review-Review relations (same user, same item, same time)
    """
    logger.info("Building edge tables...")

    edges_user_review = df[["user_id", "review_id", "timestamp"]].copy()
    edges_user_review.columns = ["src", "dst", "timestamp"]
    edges_user_review["edge_type"] = "user_writes_review"
    edges_user_review = edges_user_review[["src", "dst", "edge_type"]]

    edges_review_item = df[["review_id", "item_id", "timestamp"]].copy()
    edges_review_item.columns = ["src", "dst", "timestamp"]
    edges_review_item["edge_type"] = "review_about_item"
    edges_review_item = edges_review_item[["src", "dst", "edge_type"]]

    edges_user_item = df[["user_id", "item_id", "rating", "timestamp", "verified_purchase"]].copy()
    edges_user_item.columns = ["src", "dst", "rating", "timestamp", "verified_purchase"]
    edges_user_item["edge_type"] = "user_rates_item"
    edges_user_item = edges_user_item[["src", "dst", "edge_type"]]

    # Review-review edges
    edges_same_user = create_consecutive_edges(
        df, group_cols=["user_id"], edge_type="same_user"
    )

    edges_same_item_rating_month = create_consecutive_edges(
        df, group_cols=["item_id", "rating", "review_year_month"], edge_type="same_item_same_rating_same_month"
    )

    edges_same_item_day = create_consecutive_edges(
        df, group_cols=["item_id", "review_date"], edge_type="same_item_same_day"
    )

    edges_review_review = pd.concat(
        [edges_same_user, edges_same_item_rating_month, edges_same_item_day],
        ignore_index=True
    )

    if len(edges_review_review) > 0:
        edges_review_review = edges_review_review.drop_duplicates(
            subset=["src", "dst", "edge_type"]
        ).reset_index(drop=True)
    else:
        edges_review_review = pd.DataFrame(columns=["src", "dst", "edge_type"])

    logger.info(f"User-Review edges: {len(edges_user_review)}")
    logger.info(f"Review-Item edges: {len(edges_review_item)}")
    logger.info(f"User-Item edges: {len(edges_user_item)}")
    logger.info(f"Review-Review edges: {len(edges_review_review)}")

    return {
        "edges_user_review": edges_user_review,
        "edges_review_item": edges_review_item,
        "edges_user_item": edges_user_item,
        "edges_review_review": edges_review_review
    }

edge_tables = build_edge_tables(reviews_clean)
for name, edge_df in edge_tables.items():
    print(f"{name}: {len(edge_df)}")

## 15. Review Features

In [ ]:
def build_review_feature_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build comprehensive review feature table.
    """
    logger.info("Building review feature table...")

    feature_cols = [
        "review_id",
        # Text features
        "text_length_chars", "text_length_words", "num_exclamation", "num_question",
        "num_uppercase_words", "uppercase_word_ratio", "is_short_review",
        "is_very_short_review", "duplicate_text_count",
        # Rating features
        "rating", "is_extreme_rating", "rating_deviation_from_item_avg",
        "rating_deviation_from_user_avg",
        # Review metadata
        "helpful_vote", "verified_purchase",
        # User behavior
        "user_review_count", "user_item_count", "user_avg_rating", "user_rating_std",
        "user_active_days", "user_reviews_per_day", "user_verified_ratio",
        "user_avg_text_length", "user_singleton", "user_day_review_count",
        # Item behavior
        "item_review_count", "item_user_count", "item_avg_rating", "item_rating_std",
        "item_active_days", "item_reviews_per_day", "item_verified_ratio",
        "item_avg_text_length", "item_day_review_count", "item_month_review_count",
        "item_rating_month_count",
        # Weak label
        "suspicion_score", "weak_label"
    ]

    features = df[feature_cols].copy()
    features["verified_purchase"] = features["verified_purchase"].astype(int)

    return features

review_features = build_review_feature_table(reviews_clean)
print(f"\nReview features shape: {review_features.shape}")
print(review_features.head())

## 16. Text Embeddings (Optional)

In [ ]:
def generate_text_embeddings(
    df: pd.DataFrame,
    text_col: str = "text_raw",
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    batch_size: int = 64,
    output_path: Optional[Path] = None
) -> np.ndarray:
    """
    Generate text embeddings using SentenceTransformer.
    Optionally save to .npy file.
    """
    from sentence_transformers import SentenceTransformer

    logger.info(f"Loading model: {model_name}")
    model = SentenceTransformer(model_name)

    texts = df[text_col].fillna("").astype(str).tolist()
    logger.info(f"Generating embeddings for {len(texts)} texts...")

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    embeddings = np.array(embeddings)
    logger.info(f"Embedding shape: {embeddings.shape}")

    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(output_path, embeddings)
        logger.info(f"Saved embeddings to {output_path}")

    return embeddings

if config.GENERATE_EMBEDDINGS:
    embedding_path = config.GOLD_DIR / config.CATEGORY / f"review_text_embeddings_{config.CATEGORY}.npy"
    review_embeddings = generate_text_embeddings(
        nodes_review,
        text_col="text_raw",
        model_name=config.EMBEDDING_MODEL_NAME,
        batch_size=config.EMBEDDING_BATCH_SIZE,
        output_path=embedding_path
    )
else:
    print("Skipping embedding generation. Set config.GENERATE_EMBEDDINGS = True to run.")

## 17. Save Outputs

In [ ]:
def save_pipeline_outputs(
    category: str,
    config: PreprocessingConfig,
    reviews_clean: pd.DataFrame,
    nodes_user: pd.DataFrame,
    nodes_review: pd.DataFrame,
    nodes_item: pd.DataFrame,
    edge_tables: Dict[str, pd.DataFrame],
    review_features: pd.DataFrame
) -> Tuple[Path, Dict]:
    """
    Save all pipeline outputs to GOLD directory.
    """
    output_dir = config.GOLD_DIR / category
    output_dir.mkdir(parents=True, exist_ok=True)

    logger.info(f"Saving outputs to {output_dir}")

    # Save main tables
    reviews_clean.to_parquet(output_dir / "reviews_clean.parquet", index=False)
    nodes_user.to_parquet(output_dir / "nodes_user.parquet", index=False)
    nodes_review.to_parquet(output_dir / "nodes_review.parquet", index=False)
    nodes_item.to_parquet(output_dir / "nodes_item.parquet", index=False)

    # Save edges
    for name, edge_df in edge_tables.items():
        edge_df.to_parquet(output_dir / f"{name}.parquet", index=False)

    # Save features
    review_features.to_parquet(output_dir / "review_features.parquet", index=False)

    # Save splits
    review_splits = reviews_clean[["review_id", "split", "weak_label", "suspicion_score"]].copy()
    review_splits.to_parquet(output_dir / "review_splits.parquet", index=False)

    # Generate summary
    summary = {
        "category": category,
        "timestamp": datetime.now().isoformat(),
        "num_reviews": int(len(nodes_review)),
        "num_users": int(len(nodes_user)),
        "num_items": int(len(nodes_item)),
        "num_edges_user_review": int(len(edge_tables["edges_user_review"])),
        "num_edges_review_item": int(len(edge_tables["edges_review_item"])),
        "num_edges_user_item": int(len(edge_tables["edges_user_item"])),
        "num_edges_review_review": int(len(edge_tables["edges_review_review"])),
        "weak_label_distribution": {
            str(k): int(v) for k, v in reviews_clean["weak_label"].value_counts().to_dict().items()
        },
        "suspicion_score_distribution": {
            str(k): int(v) for k, v in reviews_clean["suspicion_score"].value_counts().sort_index().to_dict().items()
        },
        "split_distribution": {
            str(k): int(v) for k, v in reviews_clean["split"].value_counts().to_dict().items()
        }
    }

    with open(output_dir / "pipeline_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=4, ensure_ascii=False)

    logger.info(f"Successfully saved outputs to {output_dir}")

    return output_dir, summary

output_dir, summary = save_pipeline_outputs(
    category=config.CATEGORY,
    config=config,
    reviews_clean=reviews_clean,
    nodes_user=nodes_user,
    nodes_review=nodes_review,
    nodes_item=nodes_item,
    edge_tables=edge_tables,
    review_features=review_features
)

print(f"\n✓ Outputs saved to: {output_dir}")
print(f"\nSummary: {json.dumps(summary, indent=2, ensure_ascii=False)}")

## 18. Verification & Summary

In [ ]:
# List all output files
print("Output files created:")
for file in sorted(output_dir.iterdir()):
    if file.is_file():
        size = file.stat().st_size / (1024 * 1024)  # MB
        print(f"  - {file.name:<40} ({size:.2f} MB)")

In [ ]:
# Verify outputs can be read
print("\n✓ Verification - Reading saved files:")
print(f"  - Reviews: {len(pd.read_parquet(output_dir / 'reviews_clean.parquet'))} rows")
print(f"  - Users: {len(pd.read_parquet(output_dir / 'nodes_user.parquet'))} rows")
print(f"  - Items: {len(pd.read_parquet(output_dir / 'nodes_item.parquet'))} rows")
print(f"  - Review Features: {len(pd.read_parquet(output_dir / 'review_features.parquet'))} rows")
print(f"  - Review-Review Edges: {len(pd.read_parquet(output_dir / 'edges_review_review.parquet'))} rows")

In [ ]:
# Final summary
print("\n" + "="*60)
print("PREPROCESSING PIPELINE COMPLETE")
print("="*60)
print(f"\nSystem: {SYSTEM}")
print(f"Output Directory: {output_dir}")
print(f"\nStatistics:")
print(f"  Reviews: {summary['num_reviews']:,}")
print(f"  Users: {summary['num_users']:,}")
print(f"  Items: {summary['num_items']:,}")
print(f"  Total Edges: {sum(summary[k] for k in ['num_edges_user_review', 'num_edges_review_item', 'num_edges_user_item', 'num_edges_review_review']):,}")
print(f"\nLabel Distribution:")
for k, v in summary['weak_label_distribution'].items():
    pct = v / summary['num_reviews'] * 100
    print(f"  Weak label {k}: {v:,} ({pct:.1f}%)")
print("\n" + "="*60)